# 🏛️ ASKME: Fine-Tuning Open-Source FLAN-T5 for Civic Action Summarization
### AI-Powered Civic Complaint & Administrative Action Synthesis

This notebook trains and fine-tunes **`google/flan-t5-small`** (~77 Million parameters) on the **ASKME Civic Action Dataset**.

### What this model does:
When a citizen reports an issue (such as a hazardous road crater, overflowing municipal waste, or contaminated water), ASKME executes a multi-step civic workflow (Intake -> Evidence Verification -> Authority Mapping -> Formal Petition Construction -> Portal Dispatch).

The fine-tuned model synthesizes this entire telemetry stream into a concise, authoritative **Executive Administrative Summary** documenting:
1. The nature and severity of the civic issue.
2. Attached photographic exhibits and citizen evidence.
3. The exact statutory municipal authority identified and matched.
4. The formal citizen grievance reference tracking ID (`ASKME-2026-XXXXXX`).
5. Confirmation of official portal dispatch readiness for departmental inspection.

**Hardware Recommendation:** Kaggle GPU T4 (runs in ~4-6 minutes) or standard CPU.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q torch transformers datasets evaluate rouge-score accelerate pandas

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Load the ASKME Civic Dataset
We load the paired training and validation sets. If running inside Kaggle Notebook without external dataset attached, the fallback generator automatically constructs the 1,800+ paired samples inline.

In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict

# Check possible dataset locations
possible_paths = [
    ("/kaggle/input/askme-civic-summaries/askme_civic_summaries_train.csv", "/kaggle/input/askme-civic-summaries/askme_civic_summaries_val.csv"),
    ("../data/askme_civic_summaries_train.csv", "../data/askme_civic_summaries_val.csv"),
    ("data/askme_civic_summaries_train.csv", "data/askme_civic_summaries_val.csv"),
    ("./askme_civic_summaries_train.csv", "./askme_civic_summaries_val.csv")
]

train_path, val_path = None, None
for t, v in possible_paths:
    if os.path.exists(t) and os.path.exists(v):
        train_path, val_path = t, v
        break

if train_path:
    print(f"Loading dataset from: {train_path}")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
else:
    print("Generating dataset inline...")
    # Generate synthetic civic dataset inline
    import random
    random.seed(42)
    categories = ["Roads & Transportation", "Waste Management", "Water Supply & Sewage", "Street Lighting", "Public Safety"]
    authorities = ["Bruhat Bengaluru Mahanagara Palike (BBMP)", "Greater Chennai Corporation (GCC)", "Brihanmumbai Municipal Corporation (BMC)", "Municipal Corporation of Delhi (MCD)", "National Highways Authority of India (NHAI)"]
    severities = ["Moderate", "High", "Urgent"]
    
    records = []
    for i in range(1500):
        ref_id = f"ASKME-2026-{100000+i:06d}"
        cat = random.choice(categories)
        auth = random.choice(authorities)
        sev = random.choice(severities)
        photos = random.choice([0, 1, 2, 3])
        is_sent = random.choice([True, False])
        status = "Submitted" if is_sent else "Prepared"
        ev_str = f"{photos} verified photo(s) attached" if photos > 0 else "Visual evidence pending"
        
        prompt = f"summarize civic action: Issue: {cat} Defect | Category: {cat} | Severity: {sev} | Location: Sector {i%20 + 1}, Main Road | Evidence: {ev_str} | Authority: {auth} | Portal: https://grievance.gov.in | Reference ID: {ref_id} | Status: {status}"
        
        if is_sent:
            summary = f"ASKME AI analyzed the {sev.lower()}-severity {cat.lower()} grievance at Sector {i%20 + 1}, Main Road with {photos} verified photos. The matter was validated, mapped under jurisdiction of {auth}, and an official statutory representation (Ref: {ref_id}) was constructed and dispatched to the official grievance portal for urgent departmental inspection and rectification."
        else:
            summary = f"ASKME AI completed multimodal intake and triage for {cat.lower()} defect at Sector {i%20 + 1}, Main Road with {photos} verified photos. The issue was classified as {sev.lower()} severity and routed to {auth}. A formal citizen petition has been generated with reference ID {ref_id} and is prepared for transmission to the designated redressal authority."
        
        records.append({"input_text": prompt, "target_summary": summary, "category": cat, "severity": sev, "status": status})
    
    split = int(len(records) * 0.85)
    train_df = pd.DataFrame(records[:split])
    val_df = pd.DataFrame(records[split:])

print(f"Training set size:   {len(train_df):,} samples")
print(f"Validation set size: {len(val_df):,} samples")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

### Sample Data Inspection

In [ ]:
sample = dataset["train"][0]
print("=== INPUT PROMPT ===")
print(sample["input_text"])
print("\n=== TARGET EXECUTIVE SUMMARY ===")
print(sample["target_summary"])

## 3. Tokenization & Sequence Preprocessing

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "google/flan-t5-small"
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    inputs = [str(doc) for doc in examples["input_text"]]
    targets = [str(doc) for doc in examples["target_summary"]]
    
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
print("Tokenization completed successfully!")

## 4. Initialize Pretrained FLAN-T5 Model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model Architecture: {model.__class__.__name__}")
print(f"Total Parameters:   {total_params:,} ({total_params/1e6:.1f} Million)")

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

## 5. ROUGE Evaluation Metric Setup

In [ ]:
import numpy as np
import evaluate

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

## 6. Training with Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

OUTPUT_DIR = "./civic_flan_t5_finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=8 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=8 if torch.cuda.is_available() else 4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    warmup_ratio=0.1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Seq2Seq Fine-Tuning...")
train_result = trainer.train()
print(f"Training finished in {train_result.metrics['train_runtime']:.1f} seconds!")

## 7. Model Evaluation on Held-Out Validation Set

In [ ]:
eval_results = trainer.evaluate()
print("=" * 50)
print("VALIDATION BENCHMARK RESULTS:")
print("=" * 50)
for k, v in eval_results.items():
    if "rouge" in k or "loss" in k:
        print(f"{k:15}: {v}")
print("=" * 50)

## 8. Interactive Testing & Inference Verification

In [ ]:
test_prompts = [
    (
        "summarize civic action: Issue: Dangerous crater pothole causing vehicular accident | "
        "Category: Roads & Transportation | Severity: Urgent | Location: 100ft Road, Indiranagar, Bengaluru | "
        "Evidence: 2 verified photo(s) attached | Authority: Bruhat Bengaluru Mahanagara Palike (BBMP) | "
        "Portal: https://bbmp.gov.in | Reference ID: ASKME-2026-991122 | Status: Submitted"
    ),
    (
        "summarize civic action: Issue: Contaminated tap water with pungent sewage odor | "
        "Category: Water Supply & Sewage | Severity: Urgent | Location: T. Nagar, Chennai, Tamil Nadu | "
        "Evidence: 1 verified photo(s) attached | Authority: Chennai Metro Water Supply and Sewage Board (CMWSSB) | "
        "Portal: https://chennaicorporation.gov.in | Reference ID: ASKME-2026-443322 | Status: Prepared"
    )
]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

for i, p in enumerate(test_prompts, 1):
    inputs = tokenizer(p, return_tensors="pt", max_length=MAX_SOURCE_LENGTH, truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4, length_penalty=1.0)
    pred_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    
    print(f"\n--- TEST CASE #{i} ---")
    print(f"PROMPT:   {p}")
    print(f"\nOUTPUT SUMMARY:\n{pred_text}")
    print("-" * 70)

## 9. Save and Package Model for Export

In [ ]:
import shutil

# Save model weights and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model files saved in: {OUTPUT_DIR}")

# Zip for 1-click download from Kaggle output tab
shutil.make_archive("civic_flan_t5_weights", "zip", OUTPUT_DIR)
print("Successfully created: civic_flan_t5_weights.zip")
print("You can now download this zip file directly from the Kaggle Notebook Output tab!")